this notebook does cleaning of the dataset. it renames brands, removes duplicates, and saves the cleaned dataset.

In [17]:
import re
import unicodedata

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return text

    # Normalize unicode so weird look-alikes (e.g. fancy quotes) become normal ones
    text = unicodedata.normalize("NFKD", text)

    # Remove trademark, registered, copyright, emojis and symbols
    text = re.sub(r"[™©®]", "", text)

    # Remove all punctuation except allowed ones
    allowed = r"[^a-zA-Z0-9\s\-\_\/\.\,\(\)\[\]&+]"
    text = re.sub(allowed, "", text)

    # Replace multiple spaces with single space
    text = re.sub(r"\s+", " ", text)

    return text.strip()


In [18]:
from pathlib import Path

import pandas as pd

# ============================================================
# 1. Paths (assuming notebook lives in: project/notebooks/*.ipynb)
# ============================================================

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data"

# You can ignore product_names if you want, leaving it here for completeness
PRODUCT_NAMES_FILE = DATA_RAW / "insights" / "product_names.csv"
PRODUCT_NAMES_NORM_FILE = DATA_RAW / "insights" / "product_names.csv"   # in-place (if used)

PAK_FILE = DATA_RAW / "raw" / "pakistani_amazon.csv"
PAK_OUT_FILE = DATA_RAW / "raw" / "pakistani_amazon.csv"           # in-place

print("Project root:        ", PROJECT_ROOT)
print("Pakistani file:      ", PAK_FILE)
print()


# ============================================================
# 2. Brand normalization mapping
#    We will use this ONLY to replace text directly in the CSV.
# ============================================================

# Brand normalization mapping (keys: raw patterns you expect to see)
BRAND_NORMALIZATION = {
    # Case unification
    "redmi": "Xiaomi-Redmi",
    "philips": "Philips",
    "poco": "Xiaomi-Poco",
    "xiaomi": "Xiaomi",
    "samsung": "Samsung",
    "sony": "Sony",
    "lg": "LG",
    "hp": "HP",
    "dell": "Dell",
    "lenovo": "Lenovo",
    "asus": "Asus",
    "acer": "Acer",
    "canon": "Canon",
    "realme": "Realme",
    "nikon": "Nikon",
    "tp-link": "TP-Link",
    "amazonbasics": "AmazonBasics",
    "amazon basics": "AmazonBasics",
    "ikea": "IKEA",
    "mi": "Xiaomi-MI",
    # Example: if you later want families:
    # "redmi": "Xiaomi",
    # "poco": "Xiaomi",
    # "xiaomi": "Xiaomi",
}

# ============================================================
# 4. Normalize TEXT INSIDE pakistani_amazon.csv USING MAPPING
#    - No brand_guess
#    - For each mapping (raw -> canonical), search all string columns
#      and replace occurrences (case-insensitive) in-place.
# ============================================================

print("=== Normalizing text inside pakistani_amazon.csv ===")
if not PAK_FILE.exists():
    raise FileNotFoundError(f"Cannot find {PAK_FILE}")

df = pd.read_csv(PAK_FILE)

# We only touch object (string-like) columns
# obj_cols = df.select_dtypes(include=["object"]).columns.tolist()
obj_cols = ["product_name"]
print("String columns that will be normalized:", obj_cols)
print()

for col in obj_cols:
    print(f"Cleaning special characters in column: {col}")
    df[col] = df[col].astype(str).apply(clean_text)

for raw, canonical in BRAND_NORMALIZATION.items():
    raw = raw.strip()
    if not raw:
        continue

    raw_lower = raw.lower()

    total_hits = 0

    for col in obj_cols:
        s = df[col].astype(str)

        new_values = []
        col_hits = 0  # number of cells where at least one word was replaced

        for cell in s:
            text = str(cell)
            words = text.split(" ")   # split by spaces as you suggested

            replaced_here = False
            for i, w in enumerate(words):
                # compare whole word (case-insensitive)
                if w.lower() == raw_lower:
                    words[i] = canonical
                    replaced_here = True

            if replaced_here:
                col_hits += 1

            new_values.append(" ".join(words))

        if col_hits > 0:
            total_hits += col_hits
            print(
                f"Column '{col}': replacing whole word '{raw}' -> '{canonical}' "
                f"in {col_hits} cell(s)"
            )
            df[col] = new_values  # overwrite the whole column with updated text

    if total_hits == 0:
        print(f"No whole-word occurrences of '{raw}' found in any string column.")
    print()

    
# Save back in-place
df.to_csv(PAK_OUT_FILE, index=False)
print("Saved normalized Pakistani dataset to:", PAK_OUT_FILE)
print("=== Done. All replacements applied directly to the CSV ===")


Project root:         d:\Users\DELL\new-onedrive\OneDrive - Institute of Business Administration\Documents\sem7\mlops\project
Pakistani file:       d:\Users\DELL\new-onedrive\OneDrive - Institute of Business Administration\Documents\sem7\mlops\project\data\raw\pakistani_amazon.csv

=== Normalizing text inside pakistani_amazon.csv ===
String columns that will be normalized: ['product_name']

Cleaning special characters in column: product_name
No whole-word occurrences of 'redmi' found in any string column.

Column 'product_name': replacing whole word 'philips' -> 'Philips' in 22 cell(s)

No whole-word occurrences of 'poco' found in any string column.

Column 'product_name': replacing whole word 'xiaomi' -> 'Xiaomi' in 3 cell(s)

Column 'product_name': replacing whole word 'samsung' -> 'Samsung' in 46 cell(s)

Column 'product_name': replacing whole word 'sony' -> 'Sony' in 6 cell(s)

Column 'product_name': replacing whole word 'lg' -> 'LG' in 6 cell(s)

Column 'product_name': replacing w